# Gas Prices — Data Preparation
Converts yearly xlsx/xls files into a single cleaned CSV with dates as index and cities as columns.

In [ ]:
import pandas as pd
import os
import re

input_folder = 'data'

## Step 1 — Convert xlsx/xls to CSV

In [ ]:
for filename in os.listdir(input_folder):
    if filename.endswith('.xlsx') or filename.endswith('.xls'):
        filepath = os.path.join(input_folder, filename)
        df = pd.read_excel(filepath)
        csv_name = filename.replace('.xlsx', '.csv').replace('.xls', '.csv')
        df.to_csv(os.path.join(input_folder, csv_name), index=False)
        print(f'✅ Converted: {filename} → {csv_name}')

## Step 2 — Inspect CSV structure
This revealed two distinct formats:
- **Format A** (2015, 2016): dates already in header, `Unnamed: 0` as first column
- **Format B** (all others): real headers buried in row 2, first column is the year number

In [ ]:
for filename in sorted(os.listdir(input_folder)):
    if filename.endswith('.csv'):
        df = pd.read_csv(os.path.join(input_folder, filename))
        print(f'\n📄 {filename}')
        print(f'   Shape: {df.shape}')
        print(f'   Columns: {list(df.columns)}')

## Step 3 — Peek at 2014 raw structure
2014 has full datetime strings in row 2 (e.g. `2026-01-07 00:00:00`) instead of `M/D` format.
It also has **three sub-tables side by side** — visible from the `2014.1` and `2014.2` duplicate
column suffixes pandas adds. This is the root cause of the date-named NaN columns bug.

In [ ]:
df_2014_raw = pd.read_csv(os.path.join(input_folder, 'Unleaded_Retail (Incl. Tax)_WEEKLY_2014.csv'), header=None)
print(df_2014_raw.iloc[:5, :5])

## Step 4 — Helper functions

In [ ]:
def is_date(val):
    """Match M/D format (e.g. 1/7) or full datetime strings (e.g. 2026-01-07 00:00:00)"""
    val = str(val).strip()
    return bool(re.match(r'^\d{1,2}/\d{1,2}$', val)) or bool(re.match(r'^\d{4}-\d{2}-\d{2}', val))

def parse_index(index, year):
    """Handle both M/D and full datetime formats, forcing the correct year."""
    parsed = []
    for val in index:
        val = str(val).strip()
        if re.match(r'^\d{1,2}/\d{1,2}$', val):
            parsed.append(pd.to_datetime(f'{val}/{year}', format='%m/%d/%Y'))
        elif re.match(r'^\d{4}-\d{2}-\d{2}', val):
            parsed.append(pd.to_datetime(val).replace(year=int(year)))
        else:
            parsed.append(pd.NaT)
    return pd.DatetimeIndex(parsed)

def check_columns(transposed, reference_year='2023'):
    """Compare columns across all years against a reference year."""
    all_columns = {year: set(df_t.columns) for year, df_t in transposed.items()}
    reference_cols = all_columns[reference_year]
    for year, cols in sorted(all_columns.items()):
        missing = reference_cols - cols
        extra = cols - reference_cols
        if missing or extra:
            print(f'⚠️  {year} — missing: {missing} | extra: {extra}')
        else:
            print(f'✅  {year} — columns match')

## Step 5 — Load, transpose, clean and parse dates

### What was causing the NaN date columns

Some source files (notably 2014) have **multiple sub-tables placed side by side** in the same
sheet. When `pd.read_csv(header=2, index_col=0)` is applied:
- Only column 0 becomes the row index (city names for the LEFT sub-table only)
- The DATE columns of the middle/right sub-tables (e.g. col 23 for 2014's second table)
  contain date strings in row 2, so they become **column headers** with date-like names
- After `df.T`, those date-like strings end up as **column names** in `df_t`
- There was no filter to remove them, so they survived into `df_combined` as NaN-filled columns

### Fixes applied
1. **Drop date-named columns** after transposing — any column whose name passes `is_date()` is
   a parsing artefact from a multi-table layout and must be removed
2. **Drop NaT rows** after `parse_index` — safety net for any index value that couldn't be parsed
3. **Drop 'Average' from index** before `is_date` filter — 'Average' row should be excluded
   explicitly rather than relying on it failing `is_date` (it won't always)

In [ ]:
GTA_NEIGHBORHOODS = ['NORTH YORK', 'SCARBOROUGH', 'ETOBICOKE', 'BRAMPTON', 'MISSISSAUGA', 'SARNIA', 'VAUGHAN/MARKHAM']
PEEL_COLS = ['BRAMPTON', 'MISSISSAUGA']
JUNK_COLS = ['nan', 'S-Simple V-Volume Weighted P-Population Weighted', 'Average']

RENAME_2014 = {
    'WINNIPEG\xa0\xa0': 'WINNIPEG',
    'SAINTJOHN': 'SAINT JOHN',
    'STJOHNS': 'ST JOHNS',
    'Large Markets Average (P)': 'Large Markets Ave(P)',
    'Large Markets Average (S)': 'Large Markets Ave(S)',
    'Small Markets Average (P)': 'Small Markets Ave(P)',
    'Small Markets Average (S)': 'Small Markets Ave(S)',
    'Atlantic Average (P)': 'Atlantic Ave(P)',
    'Québec Average (P)': 'Quebec Ave(P)',
    'Ontario Average (P)': 'Ontario Ave(P)',
    'Western Average (P)': 'Western Ave(P)',
    'Canada Ave (V)': 'Canada Ave(V)',
    'CANADA (P)': 'Canada(P)',
    'CANADA (S)': 'Canada(S)',
}

transposed = {}

for filename in os.listdir(input_folder):
    if not filename.endswith('.csv'):
        continue

    year = filename.split('_')[-1].replace('.csv', '')
    filepath = os.path.join(input_folder, filename)

    # --- Load with correct header based on format ---
    if year in ['2015', '2016']:
        df = pd.read_csv(filepath, index_col=0)           # Format A: dates already in header
    else:
        df = pd.read_csv(filepath, header=2, index_col=0) # Format B: real header is at row 2

    # --- Transpose: cities become columns, dates become index ---
    df_t = df.T

    # --- Clean column names (cities) ---
    df_t.columns = df_t.columns.str.replace('*', '', regex=False).str.strip()

    # --- FIX 1: Drop any column whose name looks like a date.
    #     Root cause: multi-table layouts (e.g. 2014 has 3 side-by-side tables).
    #     The DATE columns of the non-leftmost sub-tables become column headers
    #     after header=2 is applied, then survive the transpose as date-named
    #     columns full of NaN. Filter them out here. ---
    df_t = df_t[[c for c in df_t.columns if not is_date(str(c))]]

    # --- Drop explicit junk columns (NaN names, footnote text, Average row) ---
    df_t = df_t.drop(columns=[c for c in df_t.columns if str(c) in JUNK_COLS], errors='ignore')

    # --- Standardize city names ---
    if year == '2014':
        df_t = df_t.rename(columns=RENAME_2014)
    if 'TORONTO' in df_t.columns:
        df_t = df_t.rename(columns={'TORONTO': 'CITY OF TORONTO'})

    # --- Compute PEEL REGION before dropping neighborhoods ---
    existing_peel = [c for c in PEEL_COLS if c in df_t.columns]
    if existing_peel:
        df_t['PEEL REGION'] = df_t[existing_peel].astype(float).mean(axis=1)

    # --- Drop GTA neighborhoods ---
    df_t = df_t.drop(columns=[c for c in GTA_NEIGHBORHOODS if c in df_t.columns], errors='ignore')

    # --- Filter rows: keep only date-like index values ---
    df_t = df_t[df_t.index.map(is_date)]

    # --- Parse index to proper datetimes, forcing the correct year ---
    df_t.index = parse_index(df_t.index, year)

    # --- FIX 2: Drop any rows that failed date parsing (NaT).
    #     Safety net for malformed or unexpected date strings. ---
    df_t = df_t[df_t.index.notna()]

    df_t.index.name = 'date'

    # --- Convert price columns to numeric (coerce any remaining junk to NaN) ---
    df_t = df_t.apply(pd.to_numeric, errors='coerce')

    transposed[year] = df_t
    print(f'✅ {year} — rows: {len(df_t)}, cols: {len(df_t.columns)}, sample: {df_t.index[:2].tolist()}')

## Step 6 — Column check
Remaining mismatches are cities that genuinely didn't exist in older datasets — expected and handled by `pd.concat` with NaN fill.

In [ ]:
check_columns(transposed, reference_year='2023')

## Step 7 — Combine and save

After concat, we do a final column audit:
- Drop any column whose name is date-like (last-resort catch for any year that slipped through)
- Report how many NaN columns remain and where

In [ ]:
df_combined = pd.concat(transposed.values()).sort_index()

# FIX 3: Final safety pass — drop any column that still looks like a date.
# This catches edge cases where a year's df_t had a date leak that wasn't caught above.
date_cols = [c for c in df_combined.columns if is_date(str(c))]
if date_cols:
    print(f'⚠️  Dropping {len(date_cols)} date-named columns that leaked through: {date_cols}')
    df_combined = df_combined.drop(columns=date_cols)
else:
    print('✅ No date-named columns found — bug is fixed!')

# Report any columns that are >50% NaN (genuine missing cities vs artefacts)
nan_report = df_combined.isna().mean().sort_values(ascending=False)
high_nan = nan_report[nan_report > 0.5]
if not high_nan.empty:
    print(f'\nColumns with >50% NaN (likely missing in older years — expected):')
    for col, pct in high_nan.items():
        print(f'   {col}: {pct:.0%} NaN')

print(f'\nShape: {df_combined.shape}')
print(f'Date range: {df_combined.index.min()} → {df_combined.index.max()}')
print(f'Columns: {len(df_combined.columns)}')
df_combined.head(3)

In [ ]:
output_path = os.path.join(input_folder, 'gas_prices_all_years.csv')
df_combined.to_csv(output_path)
print(f'✅ Saved to {output_path}')